# Bovine mutations: raw counts and amino-acid fitness effects

Influenza spilled into dairy cattle in 2024, so we have a small but growing set of
bovine branches in the global tree. This notebook looks at the **bovine** host subset to ask:

1. **Raw counts** — which amino-acid mutations recur (appear many times) on bovine branches?
2. **Distribution of fitness effects** — what does the spread of `delta_fitness` look like?
3. **Large positive effects** — which mutations are observed substantially *more* than the
   neutral model expects (large positive `delta_fitness`), especially recurrent ones?

In [1]:
import os
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

figure_dir = '../results/figures'
os.makedirs(figure_dir, exist_ok=True)

## Load the bovine subset

In [2]:
df = pd.read_csv('../results/subset_aa_fitness_effects.csv',
                 keep_default_na=False, low_memory=False)
bov = df[df['subset'] == 'bovine'].copy()
bov['label'] = bov['subtype'] + ' ' + bov['gene'] + ':' + bov['aa_mut']

# Focus on mutations actually seen on bovine branches. The file also contains
# never-observed high-opportunity mutations (actual_count == 0) whose effects are
# pinned negative by the pseudo-count; those are irrelevant to a recurrence search.
observed = bov[bov['actual_count'] >= 1].copy()
observed.head()

,subset,subset_type,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness,label
126907,bovine,host,H5,HA,HA,10,I,M,I10M,nonsynonymous,2,0.453040,0.964389,H5 HA:I10M
126908,bovine,host,H5,HA,HA,10,I,V,I10V,nonsynonymous,1,0.365969,0.549371,H5 HA:I10V
126909,bovine,host,H5,HA,HA,100,N,N,N100N,synonymous,1,0.760792,0.173725,H5 HA:N100N
126910,bovine,host,H5,HA,HA,102,A,A,A102A,synonymous,1,0.675881,0.243447,H5 HA:A102A
126911,bovine,host,H5,HA,HA,104,D,G,D104G,nonsynonymous,3,0.277058,1.505004,H5 HA:D104G


In [3]:
print('Segments with observed bovine mutations:', sorted(observed['segment'].unique()))
print('Total observed bovine AA-mutation events: '
      f"{int(observed['actual_count'].sum()):,}")
print()
print('Events per segment:')
print(observed.groupby('segment')['actual_count'].sum().astype(int)
      .sort_values(ascending=False).to_string())

Segments with observed bovine mutations: ['HA', 'MP', 'NA', 'NP', 'NS', 'PA', 'PB1', 'PB2']
Total observed bovine AA-mutation events: 3,701

Events per segment:
segment
PA     671
PB2    607
HA     522
PB1    488
NA     470
NP     414
NS     310
MP     219


In [4]:
# Summary stats of expected counts across the full bovine dataframe (incl. unobserved)
bov['expected_count'].describe()

count    3468.000000
mean        0.747313
std         0.396255
min         0.001650
25%         0.421412
50%         0.678915
75%         1.050099
max         2.208064
Name: expected_count, dtype: float64

In [5]:
# Write the observed bovine mutations in the spike (HA) gene to an untracked CSV
# (results/ is gitignored)
observed_csv = '../results/bovine_observed_mutations.csv'
observed_ha = observed[observed['gene'] == 'HA']
observed_ha.to_csv(observed_csv, index=False)
print(f'Wrote {len(observed_ha):,} HA (spike) rows to {observed_csv}')

Wrote 378 HA (spike) rows to ../results/bovine_observed_mutations.csv


## 1. Raw counts — recurrent bovine mutations

In [6]:
for k in [1, 2, 3, 4, 5, 6, 7, 8]:
    print(f'  observed >= {k} times: {(observed["actual_count"] >= k).sum():,} mutations')

  observed >= 1 times: 2,764 mutations
  observed >= 2 times: 720 mutations
  observed >= 3 times: 160 mutations
  observed >= 4 times: 40 mutations
  observed >= 5 times: 12 mutations
  observed >= 6 times: 3 mutations
  observed >= 7 times: 1 mutations
  observed >= 8 times: 1 mutations


In [7]:
top_recurrent = (
    observed.sort_values(['delta_fitness'], ascending=False)
    .head(25)
    [['segment', 'gene', 'subtype', 'aa_mut', 'mut_class',
      'actual_count', 'expected_count', 'delta_fitness']]
    .reset_index(drop=True)
)
top_recurrent

,segment,gene,subtype,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,HA,HA,H5,A172T,nonsynonymous,8,0.520125,2.120141
1,NP,NP,all,V33V,synonymous,5,0.345319,1.872789
2,PB2,PB2,all,T58A,nonsynonymous,3,0.046603,1.856795
3,NS,NEP;NS1,all,D27G;G184G,nonsynonymous,6,0.529970,1.842272
4,NA,NA,N1,Q5Q,synonymous,5,0.395302,1.815342
5,PB2,PB2,all,S181S,synonymous,4,0.311640,1.712775
6,PB2,PB2,all,T271T,synonymous,4,0.354977,1.660758
7,HA,HA,H5,V147M,nonsynonymous,6,0.804569,1.605930
8,NP,NP,all,N473S,nonsynonymous,4,0.415698,1.592146
9,NA,NA,N1,H129H,synonymous,4,0.422407,1.584846


In [8]:
top_recurrent_ha = (
    observed
    .sort_values(['delta_fitness'], ascending=False)
    .query('segment == "HA"')
    .head(25)
    [['gene', 'aa_mut', 'mut_class',
      'actual_count', 'expected_count', 'delta_fitness']]
    .reset_index(drop=True)
)
top_recurrent_ha

,gene,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,HA,A172T,nonsynonymous,8,0.520125,2.120141
1,HA,V147M,nonsynonymous,6,0.804569,1.605930
2,HA,I178R,nonsynonymous,2,0.038954,1.534415
3,HA,I352K,nonsynonymous,2,0.053596,1.507611
4,HA,D104G,nonsynonymous,3,0.277058,1.505004
5,HA,I133F,nonsynonymous,2,0.079083,1.462601
6,HA,N205K,nonsynonymous,2,0.081199,1.458952
7,HA,T143T,synonymous,3,0.331111,1.437755
8,HA,K169R,nonsynonymous,3,0.351787,1.413182
9,HA,T331T,synonymous,3,0.407306,1.350038


## 2. Distribution of fitness effects

`delta_fitness > 0` means observed **more** than the neutral model expects (tolerated /
favored); `< 0` means observed **less** than expected (deleterious). Restricted to observed
mutations and split by class (synonymous is expected to center nearer 0).

In [9]:
for name in ['nonsynonymous', 'synonymous']:
    sub = observed[observed['mut_class'] == name]
    print(f'{name}: n={len(sub):,}, median={sub["delta_fitness"].median():.3f}, '
          f'mean={sub["delta_fitness"].mean():.3f}, '
          f'min={sub["delta_fitness"].min():.3f}, max={sub["delta_fitness"].max():.3f}')

nonsynonymous: n=1,068, median=0.519, mean=0.558, min=-0.410, max=2.120
synonymous: n=1,694, median=0.455, mean=0.444, min=-0.484, max=1.873


In [10]:
fit_hist = (
    alt.Chart(observed[observed['mut_class'].isin(['nonsynonymous', 'synonymous'])])
    .mark_bar(opacity=0.6)
    .encode(
        x=alt.X('delta_fitness:Q', bin=alt.Bin(maxbins=40),
                title='delta_fitness (log observed / expected)'),
        y=alt.Y('count():Q', title='Number of observed AA mutations', stack=None),
        color=alt.Color('mut_class:N', title='Mutation class'),
        tooltip=['mut_class:N', 'count():Q'],
    )
    .properties(width=460, height=280,
                title='Distribution of bovine fitness effects (observed mutations)')
    .configure_axis(labelFontSize=12, titleFontSize=13)
    .configure_title(fontSize=14)
)
fit_hist.save(f'{figure_dir}/bovine_fitness_distribution.png', ppi=200)
fit_hist

alt.Chart(...)

## 3. Mutations with large positive fitness effects

The candidates of interest: nonsynonymous mutations observed well above the neutral
expectation. A large positive `delta_fitness` backed by a higher `actual_count` (several
independent observations) is more compelling than one driven by a single observation
against a tiny expectation — the `expected_count` column is kept so you can judge. We set a
modest recurrence floor because bovine counts are small.

In [11]:
MIN_COUNT = 3  # bovine is small; max observed recurrence is single digits

nonsyn = observed[observed['mut_class'] == 'nonsynonymous']
top_positive = (
    nonsyn[nonsyn['actual_count'] >= MIN_COUNT]
    .sort_values('delta_fitness', ascending=False)
    [['segment', 'gene', 'subtype', 'aa_mut',
      'actual_count', 'expected_count', 'delta_fitness']]
    .reset_index(drop=True)
)
print(f'Nonsynonymous mutations with actual_count >= {MIN_COUNT}: {len(top_positive)}')
top_positive.head(30)

Nonsynonymous mutations with actual_count >= 3: 51


,segment,gene,subtype,aa_mut,actual_count,expected_count,delta_fitness
0,HA,HA,H5,A172T,8,0.520125,2.120141
1,PB2,PB2,all,T58A,3,0.046603,1.856795
2,NS,NEP;NS1,all,D27G;G184G,6,0.529970,1.842272
3,HA,HA,H5,V147M,6,0.804569,1.605930
4,NP,NP,all,N473S,4,0.415698,1.592146
5,PA,PA;X-ORF,all,I201I;L11S,3,0.243424,1.549251
6,PB2,PB2,all,M483T,3,0.248107,1.542973
7,PA,PA;X-ORF,all,G235G;D45G,5,0.702091,1.520685
8,HA,HA,H5,D104G,3,0.277058,1.505004
9,PB1,PB1,all,K480R,3,0.337188,1.430470
